# LTX 200M Student Distillation (Teacher-Student)

Цель ноутбука:
- взять `Lightricks/LTX-Video` как teacher (модель из `ltx_benchmark.py`),
- автоматически собрать student-трансформер около **200M параметров**,
- обучить student на ваших портретных видео через distillation,
- сохранить student как отдельный checkpoint.

> Важно: API `diffusers` для LTX может немного отличаться между версиями. В ноутбуке есть `inspect.signature(...)` и безопасные helper-функции, чтобы адаптация была минимальной.

In [ ]:
import os
import math
import json
import inspect
import random
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from PIL import Image

from diffusers import LTXPipeline, LTXImageToVideoPipeline

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)

In [ ]:
@dataclass
class DistillConfig:
    # Teacher from ltx_benchmark.py
    teacher_model_id: str = "Lightricks/LTX-Video"
    # For pure video distillation use t2v; for image-conditioned distillation use i2v.
    pipeline_mode: str = "t2v"  # "t2v" | "i2v"
    dtype: torch.dtype = torch.bfloat16

    # Data
    dataset_root: str = "openvid_portrait_subset"
    manifest_name: str = "subset_manifest.csv"
    max_train_samples: Optional[int] = None

    # Video preprocessing (match your LTX benchmark defaults)
    frames: int = 25
    width: int = 512
    height: int = 512

    # Student target
    target_params_millions: float = 200.0

    # Training
    batch_size: int = 1
    grad_accum_steps: int = 2
    num_workers: int = 2
    epochs: int = 1
    lr: float = 1e-4
    weight_decay: float = 1e-2
    max_grad_norm: float = 1.0

    # Distillation weights
    loss_mse_weight: float = 1.0
    loss_cos_weight: float = 0.05

    # IO
    output_dir: str = "outputs_ltx_student_200m"
    save_every_steps: int = 250

cfg = DistillConfig()
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
print(asdict(cfg))

In [ ]:
def count_params_m(model: nn.Module) -> float:
    return sum(p.numel() for p in model.parameters()) / 1e6


def _find_first_config_key(config: Dict, keys: List[str]) -> Optional[str]:
    for k in keys:
        if k in config:
            return k
    return None


def _scaled_int(value: int, scale: float, minimum: int = 1, multiple_of: int = 1) -> int:
    out = max(minimum, int(round(value * scale)))
    if multiple_of > 1:
        out = max(minimum, (out // multiple_of) * multiple_of)
        if out < minimum:
            out = minimum
    return out


def make_student_from_teacher(teacher_transformer: nn.Module, target_m: float = 200.0) -> nn.Module:
    """
    Создает student из teacher.config, уменьшая ТОЛЬКО глубину (число блоков).

    Это сохраняет входной/выходной интерфейс (proj_in/proj_out), что критично для LTX,
    и резко снижает риск shape-mismatch в трансформере.
    """
    teacher_cls = teacher_transformer.__class__
    teacher_cfg = dict(teacher_transformer.config)

    layers_key = _find_first_config_key(teacher_cfg, ["num_layers", "num_hidden_layers", "layers_per_block", "depth"])
    if layers_key is None or not isinstance(teacher_cfg.get(layers_key), int):
        raise RuntimeError(f"Не удалось найти integer depth key в config: {teacher_cfg.keys()}")

    teacher_params = count_params_m(teacher_transformer)
    teacher_layers = int(teacher_cfg[layers_key])
    print(f"Teacher params: {teacher_params:.1f}M, layers={teacher_layers}")

    # Initial ratio estimate assuming params roughly linear in depth.
    ratio = max(0.05, min(1.0, target_m / max(teacher_params, 1e-6)))
    base_layers = max(2, int(round(teacher_layers * ratio)))

    # Try a small neighborhood around estimated depth and pick closest by params.
    candidates = sorted(set([max(2, base_layers + d) for d in (-2, -1, 0, 1, 2)]))

    best_model = None
    best_gap = float("inf")

    for n_layers in candidates:
        c = dict(teacher_cfg)
        c[layers_key] = int(n_layers)
        m = teacher_cls.from_config(c)
        p = count_params_m(m)
        gap = abs(p - target_m)
        print(f"  candidate layers={n_layers}: {p:.1f}M")
        if gap < best_gap:
            best_gap = gap
            best_model = m

    assert best_model is not None
    print(f"Student params: {count_params_m(best_model):.1f}M (target={target_m:.1f}M)")
    return best_model


PipelineCls = LTXPipeline if cfg.pipeline_mode == "t2v" else LTXImageToVideoPipeline
teacher_pipe = PipelineCls.from_pretrained(
    cfg.teacher_model_id,
    torch_dtype=cfg.dtype,
).to(device)
teacher_pipe.set_progress_bar_config(disable=True)

teacher_transformer = teacher_pipe.transformer.eval()
for p in teacher_transformer.parameters():
    p.requires_grad_(False)

student_transformer = make_student_from_teacher(teacher_transformer, cfg.target_params_millions).to(device)
print("Pipeline mode:", cfg.pipeline_mode)
print("Teacher class:", teacher_transformer.__class__.__name__)
print("Student class:", student_transformer.__class__.__name__)
print("Teacher in_channels (config):", getattr(teacher_transformer.config, "in_channels", None))
print("Teacher expected in_channels (inferred):", infer_expected_in_channels(teacher_transformer) if 'infer_expected_in_channels' in globals() else None)

In [ ]:
import cv2


def read_video_cv2(path: str, num_frames: int, width: int, height: int) -> torch.Tensor:
    cap = cv2.VideoCapture(path)
    frames = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (width, height), interpolation=cv2.INTER_AREA)
        frames.append(frame)
    cap.release()

    if len(frames) == 0:
        raise RuntimeError(f"empty/broken video: {path}")

    # uniform sampling to fixed frame count
    idx = np.linspace(0, len(frames) - 1, num_frames).round().astype(int)
    sampled = [frames[i] for i in idx]
    arr = np.stack(sampled).astype(np.float32) / 255.0  # [F,H,W,C]
    tensor = torch.from_numpy(arr).permute(0, 3, 1, 2)  # [F,C,H,W]
    tensor = tensor * 2.0 - 1.0
    return tensor


class PortraitVideoDataset(Dataset):
    def __init__(self, cfg: DistillConfig):
        root = Path(cfg.dataset_root)
        manifest = root / cfg.manifest_name
        videos_dir = root / "videos"

        df = pd.read_csv(manifest)
        if cfg.max_train_samples is not None:
            df = df.iloc[: cfg.max_train_samples].copy()

        self.samples = []
        for _, row in df.iterrows():
            video_file = str(row["video_file"])
            caption = str(row.get("caption", "portrait animation"))
            path = videos_dir / video_file
            if path.exists():
                self.samples.append((str(path), caption))

        self.frames = cfg.frames
        self.width = cfg.width
        self.height = cfg.height

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        path, caption = self.samples[idx]
        pix = read_video_cv2(path, self.frames, self.width, self.height)
        return {
            "pixel_values": pix,
            "caption": caption,
            "path": path,
        }


ds = PortraitVideoDataset(cfg)
print("dataset size:", len(ds))

loader = DataLoader(
    ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=True,
    drop_last=True,
)

batch = next(iter(loader))
print(batch["pixel_values"].shape, len(batch["caption"]))

In [ ]:
def encode_prompt_batch(pipe, captions: List[str], dtype: torch.dtype):
    # LTX returns: prompt_embeds, prompt_attention_mask, negative_prompt_embeds, negative_prompt_attention_mask
    with torch.no_grad():
        prompt_embeds, prompt_attention_mask, _, _ = pipe.encode_prompt(
            prompt=captions,
            negative_prompt=["" for _ in captions],
            do_classifier_free_guidance=False,
            num_videos_per_prompt=1,
            max_sequence_length=128,
            device=device,
            dtype=dtype,
        )
    return prompt_embeds, prompt_attention_mask


def encode_video_to_latents(pipe, pixel_values: torch.Tensor, dtype: torch.dtype) -> torch.Tensor:
    """
    pixel_values: [B,F,C,H,W] in [-1,1]
    returns latents: [B,C,F,H',W']

    В разных версиях LTX VAE encode ожидает либо 5D [B,C,F,H,W],
    либо 4D [B*F,C,H,W]. Поддерживаем оба формата.
    """
    b, f, c, h, w = pixel_values.shape
    x5d = pixel_values.permute(0, 2, 1, 3, 4).contiguous().to(device=device, dtype=dtype)  # [B,C,F,H,W]

    with torch.no_grad():
        try:
            encoded = pipe.vae.encode(x5d)
            if hasattr(encoded, "latent_dist"):
                z = encoded.latent_dist.sample()
            else:
                z = encoded[0] if isinstance(encoded, (tuple, list)) else encoded

            # Expected path for LTX video VAE
            if z.ndim != 5:
                raise ValueError(f"Expected 5D latents from 5D encode, got shape={tuple(z.shape)}")

        except Exception:
            # Fallback path for image-style VAEs that expect 4D inputs
            x4d = x5d.permute(0, 2, 1, 3, 4).reshape(b * f, c, h, w).contiguous()
            encoded = pipe.vae.encode(x4d)
            if hasattr(encoded, "latent_dist"):
                z = encoded.latent_dist.sample()
            else:
                z = encoded[0] if isinstance(encoded, (tuple, list)) else encoded

            if z.ndim != 4:
                raise ValueError(f"Expected 4D latents from 4D encode fallback, got shape={tuple(z.shape)}")
            z = z.reshape(b, f, z.shape[1], z.shape[2], z.shape[3]).permute(0, 2, 1, 3, 4).contiguous()

    # common scale for diffusers VAE latents
    if hasattr(pipe.vae, "config") and hasattr(pipe.vae.config, "scaling_factor"):
        z = z * pipe.vae.config.scaling_factor

    return z


def _adapt_hidden_states_channels(payload: Dict, got: int, expected: int) -> Dict:
    out = dict(payload)

    def _to_sequence_and_repeat(x: torch.Tensor) -> torch.Tensor:
        if not isinstance(x, torch.Tensor):
            return x

        # Convert to [B, N, C] when possible (LTX transformer attention expects 3D tokens).
        if x.ndim > 3:
            # channels-last path (already indicated by matmul mismatch on last dim)
            if int(x.shape[-1]) == got:
                b = x.shape[0]
                x = x.reshape(b, -1, int(x.shape[-1]))
            # channel-first fallback
            elif x.ndim == 5 and int(x.shape[1]) == got:
                b, c, f, h, w = x.shape
                x = x.permute(0, 2, 3, 4, 1).reshape(b, f * h * w, c)
            elif x.ndim == 4 and int(x.shape[1]) == got:
                b, c, h, w = x.shape
                x = x.permute(0, 2, 3, 1).reshape(b, h * w, c)

        if expected % got != 0:
            return x

        factor = expected // got

        # Repeat feature dim (last dim) for Linear input compatibility.
        if x.ndim >= 2 and int(x.shape[-1]) == got:
            reps = [1] * x.ndim
            reps[-1] = factor
            return x.repeat(*reps)

        # Rare fallback
        if x.ndim >= 2 and int(x.shape[1]) == got:
            reps = [1] * x.ndim
            reps[1] = factor
            return x.repeat(*reps)

        return x

    if "hidden_states" in out and isinstance(out["hidden_states"], torch.Tensor):
        out["hidden_states"] = _to_sequence_and_repeat(out["hidden_states"])
    if "sample" in out and isinstance(out["sample"], torch.Tensor):
        out["sample"] = _to_sequence_and_repeat(out["sample"])
    return out


def _align_token_grid(payload: Dict) -> Dict:
    out = dict(payload)
    hs = out.get("hidden_states", None)
    if not isinstance(hs, torch.Tensor) or hs.ndim != 3:
        return out

    n_tokens = int(hs.shape[1])
    nf = out.get("num_frames", None)
    hh = out.get("height", None)
    ww = out.get("width", None)

    if isinstance(nf, int) and isinstance(hh, int) and isinstance(ww, int) and nf > 0 and hh > 0 and ww > 0:
        current = nf * hh * ww
        if current == n_tokens:
            return out

        # Try to keep num_frames fixed first.
        if n_tokens % nf == 0:
            area = n_tokens // nf
            side = int(math.sqrt(area))
            if side * side == area:
                out["height"] = side
                out["width"] = side
            else:
                out["height"] = 1
                out["width"] = area
            return out

    # Robust fallback: flatten token grid into width dimension.
    out["num_frames"] = 1
    out["height"] = 1
    out["width"] = n_tokens
    return out


def call_transformer_flexible(model: nn.Module, **kwargs):
    """Передает только поддерживаемые kwargs в transformer.forward + auto-retry on shape mismatch."""
    sig = inspect.signature(model.forward)
    allowed = set(sig.parameters.keys())
    payload = {k: v for k, v in kwargs.items() if k in allowed}
    payload = _align_token_grid(payload)

    try:
        out = model(**payload)
    except RuntimeError as e:
        msg = str(e)
        # Typical linear mismatch: "mat1 and mat2 shapes cannot be multiplied (...x16 and 128x...)"
        import re
        m = re.search(r"\((?:\d+)x(\d+) and (\d+)x(?:\d+)\)", msg)
        if m and "hidden_states" in payload:
            got = int(m.group(1))
            expected = int(m.group(2))
            if expected > got and expected % got == 0:
                print(f"[retry] transformer input channel adapt: {got} -> {expected}")
                payload2 = _adapt_hidden_states_channels(payload, got=got, expected=expected)
                payload2 = _align_token_grid(payload2)
                out = model(**payload2)
            else:
                raise
        else:
            raise

    if hasattr(out, "sample"):
        return out.sample
    if isinstance(out, (tuple, list)):
        return out[0]
    return out


def sample_train_timesteps(scheduler, batch_size: int, device: torch.device) -> torch.Tensor:
    """Унифицированный сэмплинг timesteps для разных scheduler API."""
    # Prefer scheduler-provided discrete training timesteps if available.
    if hasattr(scheduler, "timesteps") and isinstance(scheduler.timesteps, torch.Tensor) and scheduler.timesteps.numel() > 0:
        idx = torch.randint(0, scheduler.timesteps.numel(), (batch_size,), device=device)
        t = scheduler.timesteps.to(device=device)[idx]
        return t

    # Fallback to integer train steps.
    n = int(getattr(scheduler.config, "num_train_timesteps", 1000))
    return torch.randint(0, n, (batch_size,), device=device, dtype=torch.long)


def add_noise_flexible(scheduler, latents: torch.Tensor, noise: torch.Tensor, timesteps: torch.Tensor) -> torch.Tensor:
    """
    Поддержка и DDPM-like (`add_noise`), и FlowMatch-like (`scale_noise`) scheduler.
    """
    if hasattr(scheduler, "add_noise"):
        return scheduler.add_noise(latents, noise, timesteps)

    if hasattr(scheduler, "scale_noise"):
        sig = inspect.signature(scheduler.scale_noise)
        params = set(sig.parameters.keys())

        # Common signature in recent diffusers: scale_noise(sample, timestep, noise)
        if {"sample", "timestep", "noise"}.issubset(params):
            return scheduler.scale_noise(sample=latents, timestep=timesteps, noise=noise)

        # Positional fallback for older variants.
        try:
            return scheduler.scale_noise(latents, timesteps, noise)
        except TypeError:
            return scheduler.scale_noise(latents, timesteps)

    raise RuntimeError(f"Unsupported scheduler API: {scheduler.__class__.__name__}")


def _call_method_flexible(obj, method_name: str, **kwargs):
    fn = getattr(obj, method_name)
    sig = inspect.signature(fn)
    allowed = set(sig.parameters.keys())
    payload = {k: v for k, v in kwargs.items() if k in allowed}
    return fn(**payload)


def infer_expected_in_channels(transformer: nn.Module) -> Optional[int]:
    # 1) config hint
    val = getattr(transformer.config, "in_channels", None)
    if isinstance(val, int) and val > 0:
        return val

    # 2) common module names
    for name in ["x_embedder", "proj_in", "input_proj", "patch_embed"]:
        mod = getattr(transformer, name, None)
        if isinstance(mod, nn.Linear):
            return int(mod.in_features)
        if hasattr(mod, "proj") and isinstance(mod.proj, nn.Linear):
            return int(mod.proj.in_features)

    # 3) generic scan for likely input projection
    for n, m in transformer.named_modules():
        if isinstance(m, nn.Linear) and any(k in n for k in ["x_embed", "proj_in", "input", "patch"]):
            return int(m.in_features)

    return None


def adapt_latents_for_transformer(pipe, transformer: nn.Module, latents: torch.Tensor) -> torch.Tensor:
    """
    Ensure latent channels match transformer expected input channels.
    """
    expected = infer_expected_in_channels(transformer)
    if expected is None:
        return latents

    got = int(latents.shape[1])
    expected = int(expected)
    if got == expected:
        return latents

    b, c, f, h, w = latents.shape

    # Try known pipeline helpers first.
    candidate_methods = ["_pack_latents", "pack_latents"]
    for name in candidate_methods:
        if hasattr(pipe, name):
            try:
                packed = _call_method_flexible(
                    pipe,
                    name,
                    latents=latents,
                    batch_size=b,
                    num_frames=f,
                    height=h,
                    width=w,
                )
                if isinstance(packed, (tuple, list)):
                    packed = packed[0]
                if isinstance(packed, torch.Tensor) and packed.ndim == 5 and int(packed.shape[1]) == expected:
                    print(f"[info] {name} adapted channels: {got} -> {expected}")
                    return packed
            except Exception:
                pass

    # Last-resort fallback: channel repeat.
    if expected % got == 0:
        factor = expected // got
        print(f"[warn] channel fallback repeat: {got} -> {expected} (x{factor})")
        return latents.repeat(1, factor, 1, 1, 1)

    raise RuntimeError(
        f"Latent channels mismatch cannot be resolved: got={got}, expected={expected}. "
        f"No compatible pack_latents helper found."
    )


print("Teacher transformer forward signature:")
print(inspect.signature(teacher_transformer.forward))

In [ ]:
optimizer = torch.optim.AdamW(
    student_transformer.parameters(),
    lr=cfg.lr,
    betas=(0.9, 0.95),
    weight_decay=cfg.weight_decay,
)

scheduler = teacher_pipe.scheduler

# Some schedulers (especially FlowMatch variants) need timesteps pre-initialized.
# For dynamic shifting, `mu` may be required.
if hasattr(scheduler, "set_timesteps"):
    num_ts = int(getattr(scheduler.config, "num_train_timesteps", 1000))
    set_ok = False

    # First attempt: standard call.
    try:
        scheduler.set_timesteps(num_ts, device=device)
        set_ok = True
    except TypeError:
        try:
            scheduler.set_timesteps(num_ts)
            set_ok = True
        except Exception:
            pass
    except ValueError:
        pass

    # FlowMatch dynamic shifting fallback.
    if not set_ok and bool(getattr(scheduler.config, "use_dynamic_shifting", False)):
        mu_candidates = [1.0, 0.5, 2.0]
        for mu in mu_candidates:
            try:
                scheduler.set_timesteps(num_ts, device=device, mu=mu)
                set_ok = True
                print(f"scheduler.set_timesteps(..., mu={mu}) succeeded")
                break
            except TypeError:
                try:
                    scheduler.set_timesteps(num_ts, mu=mu)
                    set_ok = True
                    print(f"scheduler.set_timesteps(..., mu={mu}) succeeded (no device arg)")
                    break
                except Exception:
                    continue
            except Exception:
                continue

    if not set_ok:
        print("Warning: set_timesteps could not be initialized; training will use fallback timestep sampling.")

print("scheduler:", scheduler.__class__.__name__)
print("has add_noise:", hasattr(scheduler, "add_noise"), "has scale_noise:", hasattr(scheduler, "scale_noise"))
print("dynamic shifting:", bool(getattr(scheduler.config, "use_dynamic_shifting", False)))

scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

student_transformer.train()
global_step = 0

for epoch in range(cfg.epochs):
    for step, batch in enumerate(loader):
        pixel_values = batch["pixel_values"]  # [B,F,C,H,W]
        captions = list(batch["caption"])

        bsz = pixel_values.shape[0]

        with torch.no_grad():
            latents = encode_video_to_latents(teacher_pipe, pixel_values, cfg.dtype)
            latents = adapt_latents_for_transformer(teacher_pipe, teacher_transformer, latents)
            prompt_embeds, prompt_attn = encode_prompt_batch(teacher_pipe, captions, cfg.dtype)

            noise = torch.randn_like(latents)
            timesteps = sample_train_timesteps(scheduler, bsz, device)
            noisy_latents = add_noise_flexible(scheduler, latents, noise, timesteps)
            noisy_latents = adapt_latents_for_transformer(teacher_pipe, teacher_transformer, noisy_latents)

            expected_in = infer_expected_in_channels(teacher_transformer)
            if expected_in is not None and int(noisy_latents.shape[1]) != int(expected_in):
                raise RuntimeError(
                    f"Input channel check failed before teacher forward: got={int(noisy_latents.shape[1])}, expected={int(expected_in)}"
                )

            latent_num_frames = noisy_latents.shape[2]
            latent_h = noisy_latents.shape[3]
            latent_w = noisy_latents.shape[4]

            teacher_pred = call_transformer_flexible(
                teacher_transformer,
                hidden_states=noisy_latents,
                sample=noisy_latents,
                timestep=timesteps,
                timesteps=timesteps,
                encoder_hidden_states=prompt_embeds,
                encoder_attention_mask=prompt_attn,
                num_frames=latent_num_frames,
                height=latent_h,
                width=latent_w,
                return_dict=False,
            )

        with torch.cuda.amp.autocast(enabled=(device.type == "cuda"), dtype=torch.bfloat16):
            student_pred = call_transformer_flexible(
                student_transformer,
                hidden_states=noisy_latents,
                sample=noisy_latents,
                timestep=timesteps,
                timesteps=timesteps,
                encoder_hidden_states=prompt_embeds,
                encoder_attention_mask=prompt_attn,
                num_frames=latent_num_frames,
                height=latent_h,
                width=latent_w,
                return_dict=False,
            )

            loss_mse = F.mse_loss(student_pred.float(), teacher_pred.float())
            loss_cos = 1.0 - F.cosine_similarity(
                student_pred.float().flatten(1),
                teacher_pred.float().flatten(1),
                dim=1,
            ).mean()
            loss = cfg.loss_mse_weight * loss_mse + cfg.loss_cos_weight * loss_cos
            loss = loss / cfg.grad_accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % cfg.grad_accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(student_transformer.parameters(), cfg.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

            if global_step % 10 == 0:
                print(
                    f"epoch={epoch} step={global_step} "
                    f"loss={loss.item() * cfg.grad_accum_steps:.5f} "
                    f"mse={loss_mse.item():.5f} cos={loss_cos.item():.5f}"
                )

            if global_step % cfg.save_every_steps == 0:
                ckpt_dir = Path(cfg.output_dir) / f"checkpoint-{global_step}"
                ckpt_dir.mkdir(parents=True, exist_ok=True)
                student_transformer.save_pretrained(ckpt_dir / "student_transformer")
                with (ckpt_dir / "distill_config.json").open("w", encoding="utf-8") as f:
                    json.dump(asdict(cfg), f, ensure_ascii=False, indent=2, default=str)

print("Training finished. global_step=", global_step)

In [ ]:
final_dir = Path(cfg.output_dir) / "final"
final_dir.mkdir(parents=True, exist_ok=True)

student_transformer.eval()
student_transformer.save_pretrained(final_dir / "student_transformer")

with (final_dir / "distill_config.json").open("w", encoding="utf-8") as f:
    json.dump(asdict(cfg), f, ensure_ascii=False, indent=2, default=str)

# Save a full pipeline folder with student transformer swapped in
SavePipelineCls = LTXPipeline if cfg.pipeline_mode == "t2v" else LTXImageToVideoPipeline
student_pipe = SavePipelineCls.from_pretrained(
    cfg.teacher_model_id,
    torch_dtype=cfg.dtype,
)
student_pipe.transformer = student_transformer.cpu()
student_pipe.save_pretrained(final_dir / "student_pipeline")

print("Saved:")
print(" -", final_dir / "student_transformer")
print(" -", final_dir / "student_pipeline")

## Если словили API mismatch

Если в вашей версии `diffusers` сигнатура `teacher_transformer.forward(...)` отличается:

1. Выполните ячейку с `inspect.signature(teacher_transformer.forward)`.
2. В `call_transformer_flexible(...)` уже есть фильтрация unsupported kwargs.
3. Обычно достаточно переименовать один из аргументов в train-loop:
   - `hidden_states`
   - `timestep` / `timesteps`
   - `encoder_hidden_states`
   - `encoder_attention_mask`

## Рекомендуемый старт обучения

- сначала `max_train_samples=2000`, `epochs=1` (smoke run),
- затем полный прогон на всем вашем портретном подмножестве,
- после этого сравнить teacher vs student в том же пайплайне, что в `ltx_benchmark.py` (same prompt/image/seed).